# Formating Results tables Notebook

From the stata log files to latex format.

The strategy is vary simple: read the .log files created by STATA with the results from Germany and Spain, and combine them into latex tables using Pandas.

## Preamble

Loading modules and custom functions

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sn
# import pytesseract as tb

import csv

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:,.2f}'.format
sn.set_style('whitegrid')
%matplotlib inline

In [2]:
def pass_to_csv(file_name,path_res,path_out="",keyword="Variable"):
    results_now = 0
    table_num = 0
    looking_for_sigy = False
    if path_out == "":
        path_out = path_res

    def write_output(output, table_num):
        if table_num == 0:
            fname = path_out + file_name + ".csv"
        else:
            fname = path_out + file_name + "_{}".format(table_num+1) + ".csv"
        with open(fname, 'w+') as csvfile:
            w = csv.writer(csvfile)
            w.writerows(output)
        return table_num + 1

    with open(path_res+file_name+".log") as file:
        lines = file.read().splitlines()
        output = []
        for line in lines:
            line = line.replace('|','')
            words = line.split()

            # New table header: flush any pending output first
            if len(words)>1 and words[0] == keyword:
                if output:
                    table_num = write_output(output, table_num)
                    output = []
                looking_for_sigy = False
                results_now = 1

            # Collect data rows
            if len(words)>5 and results_now:
                output.append(line.split())

            # mu_y: stop collecting main rows, wait for optional sig_y
            if len(words)>5 and words[0] == "mu_y":
                results_now = 0
                looking_for_sigy = True

            # Optional sig_y line immediately after mu_y (Spain v21 only)
            # Exclude the mu_y line itself (already handled above)
            if looking_for_sigy and len(words)>5 and words[0] != "mu_y":
                if words[0] == "sig_y":
                    output.append(line.split())
                looking_for_sigy = False
                table_num = write_output(output, table_num)
                output = []

        # Flush any remaining output (e.g. Germany logs that end after mu_y)
        if output:
            table_num = write_output(output, table_num)

def first_format(dataset,label):
    dataset['N'] = dataset.loc['Obs'].iloc[0]
    dataset['N'] = dataset['N'].apply(lambda x: x.replace(',',''))
    dataset['N'] = dataset['N'].apply(lambda x: float(x)/2).apply(lambda x: "{:,.0f}".format(float(x)))
    dataset = dataset.rename(index={'Mean':label})
    dataset = dataset.loc[label].copy()
    return dataset

def format_res_table(tables, include_sd=False):
    names = ['lsig_y2','sig_x','lsig_c','lsig_e2','lsig_b2','lpsig_c','lpsig_x','lpsig_e','lpsig_b','N','mu_y','sig_y']
    
    table_res = pd.concat(tables,axis=1).T
    
    old_names_var = ['lsig_y2','sig_x','lsig_c','lsig_e2','lsig_b2','lpsig_c','lpsig_x','lpsig_e','lpsig_b','mu_y','sig_y']
    new_names_var = [r'$\sigma^{2}_y$',r'$\sigma^{2}_n$',r'$\sigma^{2}_c$',r'$\sigma^{2}_e$',r'$\sigma^{2}_b$',
                     r'% constant',r'% spell number','% duration dependence','% heterogeneity','Average duration (days)','Std. duration (days)']
    reorder_cols = [r'$\sigma^{2}_y$',r'$\sigma^{2}_c$',r'$\sigma^{2}_n$',r'$\sigma^{2}_e$',r'$\sigma^{2}_b$',
                     r'% constant',r'% spell number','% duration dependence','% heterogeneity','N','Average duration (days)']
    if include_sd:
        reorder_cols.append('Std. duration (days)')
    
    table_res.rename(columns=dict(zip(old_names_var,new_names_var)),inplace='True')
    table_res = table_res.reindex(columns=reorder_cols)
    
    # sig_y is variance in levels; convert to SD (only when included)
    if include_sd:
        table_res['Std. duration (days)'] = table_res['Std. duration (days)'].apply(lambda x: np.sqrt(float(x)))
    
    table_res = table_res.T

    return table_res

## 1 Baseline results

### 1.1 General - Germany

In [3]:
path_res = "./data/intermediate/germany/"

# Baseline decomp
pass_to_csv("031_decomp_2011",path_res)
ger_res = pd.read_csv(path_res+"031_decomp_2011.csv",index_col=0).T
ger_res = first_format(ger_res,'Germany')
# SD in levels from 05_descriptives-2.log (tabstat days, baseline sample)
ger_res['sig_y'] = 253701.2
# No outliers
pass_to_csv("032_decomp",path_res)
ger_res_noout = pd.read_csv(path_res+"032_decomp.csv",index_col=0).T
ger_res_noout = first_format(ger_res_noout,'Germany')

### Spain

In [4]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"
# Baseline decomp
pass_to_csv("results2011_T_harm",path_res,path_out = path_out)
spa_res_11 = pd.read_csv(path_out+"results2011_T_harm.csv",index_col=0).T
spa_res_11 = first_format(spa_res_11,'Spain')
# No outliers
pass_to_csv("results2011_noout_T_harm",path_res,path_out = path_out)
spa_res_11_noout = pd.read_csv(path_out+"results2011_noout_T_harm.csv",index_col=0).T
spa_res_11_noout = first_format(spa_res_11_noout,'Spain')

In [5]:
format_res_table([ger_res,spa_res_11], include_sd=True)

,Germany,Spain
Variable,,
$\sigma^{2}_y$,2.57,1.86
$\sigma^{2}_c$,1.64,1.64
$\sigma^{2}_n$,0.04,0.10
$\sigma^{2}_e$,0.69,-0.05
$\sigma^{2}_b$,0.19,0.16
% constant,0.64,0.83
% spell number,0.02,0.05
% duration dependence,0.27,-0.03
% heterogeneity,0.07,0.09


### 1.2 No outliers

In [6]:
print(format_res_table([ger_res_noout,spa_res_11_noout]).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lll}
\toprule
 & Germany & Spain \\
Variable &  &  \\
\midrule
$\sigma^{2}_y$ & 2.563 & 1.792 \\
$\sigma^{2}_c$ & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.042 & 0.074 \\
$\sigma^{2}_e$ & 0.690 & -0.109 \\
$\sigma^{2}_b$ & 0.186 & 0.182 \\
% constant & 0.642 & 0.796 \\
% spell number & 0.016 & 0.041 \\
% duration dependence & 0.269 & -0.061 \\
% heterogeneity & 0.072 & 0.102 \\
N & 81,656 & 64,107 \\
Average duration (days) & 357.992 & 803.791 \\
\bottomrule
\end{tabular}



### 1.3 Different Time Windows

Spain

In [7]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"
years = [2011, 2005, 1999, 1992]
tables = []
tables_noout = []

for year in years:
    # Baseline decomp
    pass_to_csv("results{}_T_harm".format(year),path_res,path_out = path_out)
    spa_res_x = pd.read_csv(path_out+"results{}_T_harm.csv".format(year),index_col=0).T
    spa_res_x = first_format(spa_res_x,str(year))
    tables.append(spa_res_x)
    # No outliers
    pass_to_csv("results{}_noout_T_harm".format(year),path_res,path_out = path_out)
    spa_res_x_noout = pd.read_csv(path_out+"results{}_noout_T_harm.csv".format(year),index_col=0).T
    spa_res_x_noout = first_format(spa_res_x_noout,str(year))
    tables_noout.append(spa_res_x_noout)

In [8]:
print(format_res_table(tables, include_sd=True))

                            2011     2005     1999     1992
Variable                                                   
$\sigma^{2}_y$              1.86     1.88     1.87     2.08
$\sigma^{2}_c$              1.64     1.64     1.64     1.64
$\sigma^{2}_n$              0.10     0.01     0.01     0.00
$\sigma^{2}_e$             -0.05     0.02    -0.00     0.10
$\sigma^{2}_b$              0.16     0.21     0.22     0.33
% constant                  0.83     0.87     0.88     0.79
% spell number              0.05     0.00     0.00     0.00
% duration dependence      -0.03     0.01    -0.00     0.05
% heterogeneity             0.09     0.11     0.12     0.16
N                         69,406  126,285  165,237  215,907
Average duration (days)   912.45 1,023.84 1,063.45   927.49
Std. duration (days)    1,047.16 1,167.79 1,195.19 1,187.57


In [9]:
print(format_res_table(tables, include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & 2011 & 2005 & 1999 & 1992 \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 1.855 & 1.880 & 1.872 & 2.081 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.098 & 0.009 & 0.008 & 0.001 \\
$\sigma^{2}_e$ & -0.049 & 0.016 & -0.002 & 0.103 \\
$\sigma^{2}_b$ & 0.162 & 0.211 & 0.221 & 0.332 \\
% constant & 0.833 & 0.875 & 0.877 & 0.791 \\
% spell number & 0.053 & 0.005 & 0.004 & 0.000 \\
% duration dependence & -0.027 & 0.009 & -0.001 & 0.050 \\
% heterogeneity & 0.087 & 0.112 & 0.118 & 0.159 \\
N & 69,406 & 126,285 & 165,237 & 215,907 \\
Average duration (days) & 912.447 & 1023.842 & 1063.454 & 927.491 \\
Std. duration (days) & 1047.165 & 1167.795 & 1195.188 & 1187.574 \\
\bottomrule
\end{tabular}



Germany

In [10]:
path_res = "./data/intermediate/germany/"
path_out = "./data/intermediate/germany/"

files = [
    ("031_decomp_2011", "2011"),
    ("031_decomp_2014", "2014"),
    ("031_decomp_2016", "2016"),
]

tables_1416 = []

for stub, year in files:
    pass_to_csv(stub, path_res, path_out=path_out)
    df = pd.read_csv(f"{path_out}{stub}.csv", index_col=0).T
    df = first_format(df, year)
    tables_1416.append(df)

In [11]:
format_res_table(tables_1416, include_sd=True)

,2011,2014,2016
Variable,,,
$\sigma^{2}_y$,2.57,2.47,2.36
$\sigma^{2}_c$,1.64,1.64,1.64
$\sigma^{2}_n$,0.04,0.02,0.01
$\sigma^{2}_e$,0.69,0.64,0.55
$\sigma^{2}_b$,0.19,0.16,0.15
% constant,0.64,0.67,0.70
% spell number,0.02,0.01,0.01
% duration dependence,0.27,0.26,0.23
% heterogeneity,0.07,0.07,0.06


In [12]:
print(
    format_res_table(
        tables_1416,
        include_sd=True
    ).to_latex(
        escape=False,
        float_format="%.3f"
    )
)

\begin{tabular}{llll}
\toprule
 & 2011 & 2014 & 2016 \\
Variable &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.568 & 2.471 & 2.363 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.043 & 0.022 & 0.015 \\
$\sigma^{2}_e$ & 0.694 & 0.643 & 0.552 \\
$\sigma^{2}_b$ & 0.186 & 0.161 & 0.152 \\
% constant & 0.641 & 0.666 & 0.696 \\
% spell number & 0.017 & 0.009 & 0.006 \\
% duration dependence & 0.270 & 0.260 & 0.234 \\
% heterogeneity & 0.072 & 0.065 & 0.064 \\
N & 81,768 & 48,061 & 27,601 \\
Average duration (days) & 360.175 & 282.948 & 229.044 \\
Std. duration (days) & 494.538 & 379.917 & 306.809 \\
\bottomrule
\end{tabular}



## 2. Education

### Germany

In [13]:
path_res = "./data/intermediate/germany/"
ed_levels = ["college","higher","secondary","primary"]
res_ger_edu = []

for edu in ed_levels:
    pass_to_csv("036_decomp_"+edu,path_res)
    ger_res = pd.read_csv(path_res+"036_decomp_{}.csv".format(edu),index_col=0).T
    ger_res = first_format(ger_res,edu)
    res_ger_edu.append(ger_res)


In [14]:
format_res_table(res_ger_edu, include_sd=True)

,college,higher,secondary,primary
Variable,,,,
$\sigma^{2}_y$,2.51,2.55,2.52,2.91
$\sigma^{2}_c$,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.15,0.02,0.02,0.17
$\sigma^{2}_e$,0.67,0.70,0.60,0.86
$\sigma^{2}_b$,0.04,0.19,0.25,0.25
% constant,0.66,0.64,0.65,0.56
% spell number,0.06,0.01,0.01,0.06
% duration dependence,0.27,0.27,0.24,0.29
% heterogeneity,0.02,0.07,0.10,0.08


### Spain

In [15]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"
res_spa_edu = []

for edu in ed_levels:
    pass_to_csv("results2011_{}_T_harm".format(edu),path_res,path_out = path_out)
    spa_res = pd.read_csv(path_out+"results2011_{}_T_harm.csv".format(edu),index_col=0).T
    spa_res = first_format(spa_res,edu)
    res_spa_edu.append(spa_res)


In [16]:
format_res_table(res_spa_edu, include_sd=True)

,college,higher,secondary,primary
Variable,,,,
$\sigma^{2}_y$,1.85,1.76,1.84,1.98
$\sigma^{2}_c$,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.05,0.10,0.12,0.11
$\sigma^{2}_e$,-0.03,-0.11,-0.05,0.04
$\sigma^{2}_b$,0.18,0.13,0.12,0.18
% constant,0.86,0.80,0.84,0.83
% spell number,0.03,0.06,0.07,0.06
% duration dependence,-0.02,-0.06,-0.03,0.02
% heterogeneity,0.10,0.08,0.07,0.09


In [17]:
print(format_res_table(res_ger_edu, include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & college & higher & secondary & primary \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.508 & 2.554 & 2.516 & 2.914 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.152 & 0.022 & 0.022 & 0.167 \\
$\sigma^{2}_e$ & 0.667 & 0.699 & 0.600 & 0.856 \\
$\sigma^{2}_b$ & 0.044 & 0.188 & 0.249 & 0.246 \\
% constant & 0.656 & 0.644 & 0.654 & 0.565 \\
% spell number & 0.061 & 0.009 & 0.009 & 0.057 \\
% duration dependence & 0.266 & 0.274 & 0.238 & 0.294 \\
% heterogeneity & 0.018 & 0.074 & 0.099 & 0.085 \\
N & 16,798 & 43,785 & 16,437 & 1,749 \\
Average duration (days) & 429.743 & 349.011 & 325.666 & 435.132 \\
Std. duration (days) & 543.028 & 477.321 & 474.211 & 605.592 \\
\bottomrule
\end{tabular}



In [18]:
print(format_res_table(res_spa_edu, include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & college & higher & secondary & primary \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 1.850 & 1.764 & 1.839 & 1.976 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.054 & 0.100 & 0.123 & 0.110 \\
$\sigma^{2}_e$ & -0.028 & -0.114 & -0.052 & 0.040 \\
$\sigma^{2}_b$ & 0.179 & 0.133 & 0.123 & 0.181 \\
% constant & 0.859 & 0.804 & 0.838 & 0.833 \\
% spell number & 0.029 & 0.057 & 0.067 & 0.056 \\
% duration dependence & -0.015 & -0.064 & -0.028 & 0.020 \\
% heterogeneity & 0.097 & 0.075 & 0.067 & 0.092 \\
N & 14,908 & 17,766 & 26,847 & 8,240 \\
Average duration (days) & 723.076 & 880.017 & 999.712 & 1043.674 \\
Std. duration (days) & 848.861 & 972.619 & 1116.282 & 1231.377 \\
\bottomrule
\end{tabular}



## 3. Sex

### Germany

In [19]:
path_res = "./data/intermediate/germany/"
sexes = ['female','male']
res_ger_sex = []

pass_to_csv("033_decomp",path_res)
ger_res_fem = pd.read_csv(path_res+"033_decomp_2.csv",index_col=0).T
ger_res_fem = first_format(ger_res_fem,"female")
ger_res_mal = pd.read_csv(path_res+"033_decomp.csv",index_col=0).T
ger_res_mal = first_format(ger_res_mal,"male")


### Spain

In [20]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"

pass_to_csv("results2011_fem_T_harm",path_res,path_out = path_out)
spa_res_fem = pd.read_csv(path_out+"results2011_fem_T_harm.csv".format(edu),index_col=0).T
spa_res_fem = first_format(spa_res_fem,"female")

pass_to_csv("results2011_mal_T_harm",path_res,path_out = path_out)
spa_res_mal = pd.read_csv(path_out+"results2011_mal_T_harm.csv".format(edu),index_col=0).T
spa_res_mal = first_format(spa_res_mal,"male")

In [21]:
format_res_table([ger_res_fem,ger_res_mal,spa_res_fem,spa_res_mal], include_sd=True)

,female,male,female,male
Variable,,,,
$\sigma^{2}_y$,2.27,2.82,1.82,1.88
$\sigma^{2}_c$,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.02,0.08,0.09,0.11
$\sigma^{2}_e$,0.60,0.77,-0.06,-0.04
$\sigma^{2}_b$,0.01,0.33,0.14,0.18
% constant,0.72,0.58,0.84,0.83
% spell number,0.01,0.03,0.05,0.06
% duration dependence,0.26,0.27,-0.03,-0.02
% heterogeneity,0.01,0.12,0.08,0.09


In [22]:
print(format_res_table([ger_res_fem,ger_res_mal,spa_res_fem,spa_res_mal], include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & female & male & female & male \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.272 & 2.818 & 1.818 & 1.884 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.018 & 0.077 & 0.086 & 0.108 \\
$\sigma^{2}_e$ & 0.597 & 0.770 & -0.058 & -0.044 \\
$\sigma^{2}_b$ & 0.012 & 0.326 & 0.144 & 0.176 \\
% constant & 0.724 & 0.584 & 0.842 & 0.826 \\
% spell number & 0.008 & 0.027 & 0.047 & 0.057 \\
% duration dependence & 0.263 & 0.273 & -0.032 & -0.023 \\
% heterogeneity & 0.005 & 0.116 & 0.079 & 0.093 \\
N & 40,691 & 41,077 & 29,774 & 39,632 \\
Average duration (days) & 378.578 & 341.946 & 899.500 & 922.174 \\
Std. duration (days) & 494.494 & 493.910 & 1033.914 & 1056.912 \\
\bottomrule
\end{tabular}



### 3.1 Spain, different time windows by sex

In [23]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"
years = [2011, 2005, 1999, 1992]
tables_f = []
tables_m = []

for year in years:
    pass_to_csv("results{}_fem_T_harm".format(year),path_res,path_out = path_out)
    spa_res_fem_i = pd.read_csv(path_out+"results{}_fem_T_harm.csv".format(year),index_col=0).T
    spa_res_fem_i = first_format(spa_res_fem_i,year)
    tables_f.append(spa_res_fem_i)
    
    pass_to_csv("results{}_mal_T_harm".format(year),path_res,path_out = path_out)
    spa_res_mal_i = pd.read_csv(path_out+"results{}_mal_T_harm.csv".format(year),index_col=0).T
    spa_res_mal_i = first_format(spa_res_mal_i,year)
    tables_m.append(spa_res_mal_i)

In [24]:
format_res_table(tables_f+tables_m, include_sd=True)

,2011,2005,1999,1992,2011,2005,1999,1992
Variable,,,,,,,,
$\sigma^{2}_y$,1.82,1.85,1.89,2.04,1.88,1.90,1.86,2.08
$\sigma^{2}_c$,1.64,1.64,1.64,1.64,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.09,0.01,0.01,0.00,0.11,0.01,0.01,0.00
$\sigma^{2}_e$,-0.06,0.02,0.03,0.13,-0.04,0.01,-0.03,0.08
$\sigma^{2}_b$,0.14,0.18,0.21,0.26,0.18,0.23,0.23,0.35
% constant,0.84,0.89,0.87,0.81,0.83,0.86,0.86,0.79
% spell number,0.05,0.00,0.00,0.00,0.06,0.01,0.01,0.00
% duration dependence,-0.03,0.01,0.02,0.07,-0.02,0.01,-0.01,0.04
% heterogeneity,0.08,0.10,0.11,0.13,0.09,0.12,0.12,0.17


In [25]:
print(format_res_table(tables_f+tables_m, include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllllllll}
\toprule
 & 2011 & 2005 & 1999 & 1992 & 2011 & 2005 & 1999 & 1992 \\
Variable &  &  &  &  &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 1.818 & 1.851 & 1.891 & 2.042 & 1.884 & 1.903 & 1.857 & 2.081 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.086 & 0.007 & 0.005 & 0.000 & 0.108 & 0.010 & 0.009 & 0.002 \\
$\sigma^{2}_e$ & -0.058 & 0.018 & 0.031 & 0.134 & -0.044 & 0.015 & -0.028 & 0.082 \\
$\sigma^{2}_b$ & 0.144 & 0.181 & 0.209 & 0.262 & 0.176 & 0.233 & 0.231 & 0.353 \\
% constant & 0.842 & 0.889 & 0.870 & 0.806 & 0.826 & 0.864 & 0.856 & 0.790 \\
% spell number & 0.047 & 0.004 & 0.003 & 0.000 & 0.057 & 0.005 & 0.005 & 0.001 \\
% duration dependence & -0.032 & 0.009 & 0.017 & 0.066 & -0.023 & 0.008 & -0.015 & 0.039 \\
% heterogeneity & 0.079 & 0.098 & 0.111 & 0.128 & 0.093 & 0.122 & 0.124 & 0.170 \\
N & 29,774 & 54,820 & 71,715 & 88,446 & 39,632 & 71,465 & 93,522 & 127,461 \\
Average duration (days) & 899.500

## 3. Age

### Germany

In [26]:
path_res = "./data/intermediate/germany/"

pass_to_csv("034_decomp_25_35",path_res)
ger_res_25_35 = pd.read_csv(path_res+"034_decomp_25_35.csv",index_col=0).T
ger_res_25_35 = first_format(ger_res_25_35,'25-35')

pass_to_csv("034_decomp_35_55",path_res)
ger_res_35_55 = pd.read_csv(path_res+"034_decomp_35_55.csv",index_col=0).T
ger_res_35_55 = first_format(ger_res_35_55,'35-55')


### Spain

In [27]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"

pass_to_csv("results2011_2535_T_harm",path_res,path_out = path_out)
spa_res_25_35 = pd.read_csv(path_out+"results2011_2535_T_harm.csv",index_col=0).T
spa_res_25_35 = first_format(spa_res_25_35,'25-35')
# No outliers
pass_to_csv("results2011_3555_T_harm",path_res,path_out = path_out)
spa_res_35_55 = pd.read_csv(path_out+"results2011_3555_T_harm.csv",index_col=0).T
spa_res_35_55 = first_format(spa_res_35_55,'35-55')

In [28]:
format_res_table([ger_res_25_35,ger_res_35_55,spa_res_25_35,spa_res_35_55], include_sd=True)

,25-35,35-55,25-35,35-55
Variable,,,,
$\sigma^{2}_y$,2.51,2.50,1.66,1.80
$\sigma^{2}_c$,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.05,0.02,0.00,0.07
$\sigma^{2}_e$,0.76,0.47,-0.19,-0.08
$\sigma^{2}_b$,0.07,0.37,0.20,0.15
% constant,0.65,0.66,0.76,0.83
% spell number,0.02,0.01,0.00,0.04
% duration dependence,0.30,0.19,-0.12,-0.04
% heterogeneity,0.03,0.15,0.12,0.09


In [29]:
print(format_res_table([ger_res_25_35,ger_res_35_55,spa_res_25_35,spa_res_35_55], include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & 25-35 & 35-55 & 25-35 & 35-55 \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.514 & 2.497 & 1.657 & 1.797 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.046 & 0.017 & 0.003 & 0.075 \\
$\sigma^{2}_e$ & 0.757 & 0.469 & -0.191 & -0.078 \\
$\sigma^{2}_b$ & 0.066 & 0.366 & 0.200 & 0.155 \\
% constant & 0.654 & 0.659 & 0.762 & 0.829 \\
% spell number & 0.018 & 0.007 & 0.002 & 0.042 \\
% duration dependence & 0.301 & 0.188 & -0.115 & -0.043 \\
% heterogeneity & 0.026 & 0.146 & 0.121 & 0.086 \\
N & 54,986 & 19,730 & 23,866 & 27,643 \\
Average duration (days) & 381.364 & 271.504 & 631.152 & 838.058 \\
Std. duration (days) & 488.420 & 437.776 & 610.922 & 863.063 \\
\bottomrule
\end{tabular}



In [30]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"
years = [2011, 2005, 1999, 1992]
tables_2535 = []
tables_3555 = []

for year in years:
    pass_to_csv("results{}_2535_T_harm".format(year),path_res,path_out = path_out)
    spa_res_2535 = pd.read_csv(path_out+"results{}_2535_T_harm.csv".format(year),index_col=0).T
    spa_res_2535 = first_format(spa_res_2535,year)
    tables_2535.append(spa_res_2535)
    
    pass_to_csv("results{}_3555_T_harm".format(year),path_res,path_out = path_out)
    spa_res_3555 = pd.read_csv(path_out+"results{}_3555_T_harm.csv".format(year),index_col=0).T
    spa_res_3555 = first_format(spa_res_3555,year)
    tables_3555.append(spa_res_3555)

In [31]:
print(format_res_table(tables_2535+tables_3555, include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllllllll}
\toprule
 & 2011 & 2005 & 1999 & 1992 & 2011 & 2005 & 1999 & 1992 \\
Variable &  &  &  &  &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 1.657 & 1.684 & 1.680 & 1.930 & 1.797 & 1.875 & 1.930 & 2.132 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.003 & 0.013 & 0.018 & 0.036 & 0.075 & 0.001 & 0.003 & 0.002 \\
$\sigma^{2}_e$ & -0.191 & -0.167 & -0.168 & -0.036 & -0.078 & 0.021 & 0.036 & 0.079 \\
$\sigma^{2}_b$ & 0.200 & 0.193 & 0.185 & 0.284 & 0.155 & 0.208 & 0.246 & 0.406 \\
% constant & 0.762 & 0.778 & 0.779 & 0.816 & 0.829 & 0.877 & 0.852 & 0.772 \\
% spell number & 0.002 & 0.008 & 0.011 & 0.019 & 0.042 & 0.001 & 0.001 & 0.001 \\
% duration dependence & -0.115 & -0.099 & -0.100 & -0.018 & -0.043 & 0.011 & 0.019 & 0.037 \\
% heterogeneity & 0.121 & 0.115 & 0.110 & 0.147 & 0.086 & 0.111 & 0.127 & 0.190 \\
N & 23,866 & 49,722 & 69,243 & 104,790 & 27,643 & 40,031 & 42,474 & 54,514 \\
Average duration (days) & 631

In [32]:
format_res_table(tables_2535+tables_3555, include_sd=True)

,2011,2005,1999,1992,2011,2005,1999,1992
Variable,,,,,,,,
$\sigma^{2}_y$,1.66,1.68,1.68,1.93,1.80,1.87,1.93,2.13
$\sigma^{2}_c$,1.64,1.64,1.64,1.64,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.00,0.01,0.02,0.04,0.07,0.00,0.00,0.00
$\sigma^{2}_e$,-0.19,-0.17,-0.17,-0.04,-0.08,0.02,0.04,0.08
$\sigma^{2}_b$,0.20,0.19,0.19,0.28,0.15,0.21,0.25,0.41
% constant,0.76,0.78,0.78,0.82,0.83,0.88,0.85,0.77
% spell number,0.00,0.01,0.01,0.02,0.04,0.00,0.00,0.00
% duration dependence,-0.12,-0.10,-0.10,-0.02,-0.04,0.01,0.02,0.04
% heterogeneity,0.12,0.11,0.11,0.15,0.09,0.11,0.13,0.19


Age controls

In [33]:
path_res = "./data/intermediate/germany/"

# Baseline decomp
pass_to_csv("031_decomp_2011",path_res)
ger_res = pd.read_csv(path_res+"031_decomp_2011.csv",index_col=0).T
ger_res = first_format(ger_res,'Baseline')
# Age controls
pass_to_csv("035_decomp",path_res)
ger_res_age = pd.read_csv(path_res+"035_decomp.csv",index_col=0).T
ger_res_age = first_format(ger_res_age,r'Age controls')

In [34]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"

# Baseline decomp
pass_to_csv("results2011_T_harm",path_res,path_out = path_out)
spa_res_11 = pd.read_csv(path_out+"results2011_T_harm.csv",index_col=0).T
spa_res_11 = first_format(spa_res_11,'Baseline')
# Age controls
pass_to_csv("results2011_age_T_harm",path_res,path_out = path_out)
spa_res_age = pd.read_csv(path_out+"results2011_age_T_harm.csv",index_col=0).T
spa_res_age = first_format(spa_res_age,r'Age controls')

In [35]:
print(format_res_table([ger_res,ger_res_age,spa_res_11,spa_res_age]).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & Baseline & Age controls & Baseline & Age controls \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.568 & 2.568 & 1.855 & 1.855 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.043 & 0.086 & 0.098 & 0.105 \\
$\sigma^{2}_e$ & 0.694 & 0.676 & -0.049 & -0.064 \\
$\sigma^{2}_b$ & 0.186 & 0.161 & 0.162 & 0.169 \\
% constant & 0.641 & 0.641 & 0.833 & 0.818 \\
% spell number & 0.017 & 0.033 & 0.053 & 0.057 \\
% duration dependence & 0.270 & 0.263 & -0.027 & -0.034 \\
% heterogeneity & 0.072 & 0.063 & 0.087 & 0.091 \\
N & 81,768 & 81,768 & 69,406 & 69,406 \\
Average duration (days) & 360.175 & 360.175 & 912.447 & 912.447 \\
\bottomrule
\end{tabular}



## 4. Business Cycle Controls

In [36]:
path_res = "./data/intermediate/germany/"

# Baseline decomp
pass_to_csv("031_decomp_2011",path_res)
ger_res = pd.read_csv(path_res+"031_decomp_2011.csv",index_col=0).T
ger_res = first_format(ger_res,'Baseline')
# GDP Growth
pass_to_csv("037_decomp_growth_unemp",path_res)
ger_res_unemp = pd.read_csv(path_res+"037_decomp_growth_unemp.csv",index_col=0).T
ger_res_unemp = first_format(ger_res_unemp,r'$\Delta$u')
# Unemployment rate change
pass_to_csv("037_decomp_real_gdp_growth",path_res)
ger_res_gdp = pd.read_csv(path_res+"037_decomp_real_gdp_growth.csv",index_col=0).T
ger_res_gdp = first_format(ger_res_gdp,r'$\Delta$GDP')


In [37]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"

# Baseline decomp
pass_to_csv("results2011_T_harm",path_res,path_out = path_out)
spa_res_11 = pd.read_csv(path_out+"results2011_T_harm.csv",index_col=0).T
spa_res_11 = first_format(spa_res_11,'Baseline')
# Unemployment rate change
pass_to_csv("results2011_unemp_T_harm",path_res,path_out = path_out)
spa_res_unemp = pd.read_csv(path_out+"results2011_unemp_T_harm.csv",index_col=0).T
spa_res_unemp = first_format(spa_res_unemp,r'$\Delta$u')
# GDP Growth
pass_to_csv("results2011_dY_T_harm",path_res,path_out = path_out)
spa_res_gdp = pd.read_csv(path_out+"results2011_dY_T_harm.csv",index_col=0).T
spa_res_gdp = first_format(spa_res_gdp,r'$\Delta$GDP')

In [38]:
format_res_table([ger_res,ger_res_unemp,ger_res_gdp,
                        spa_res_11,spa_res_unemp,spa_res_gdp])

,Baseline,$\Delta$u,$\Delta$GDP,Baseline,$\Delta$u,$\Delta$GDP
Variable,,,,,,
$\sigma^{2}_y$,2.57,2.57,2.57,1.86,1.86,1.86
$\sigma^{2}_c$,1.64,1.64,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.04,0.07,0.06,0.10,0.14,0.10
$\sigma^{2}_e$,0.69,0.66,0.68,-0.05,-0.08,-0.05
$\sigma^{2}_b$,0.19,0.19,0.19,0.16,0.15,0.16
% constant,0.64,0.64,0.64,0.83,0.80,0.83
% spell number,0.02,0.03,0.02,0.05,0.08,0.05
% duration dependence,0.27,0.26,0.26,-0.03,-0.04,-0.03
% heterogeneity,0.07,0.08,0.07,0.09,0.08,0.09


In [39]:
print(format_res_table([ger_res,ger_res_unemp,ger_res_gdp,
                        spa_res_11,spa_res_unemp,spa_res_gdp]).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllllll}
\toprule
 & Baseline & $\Delta$u & $\Delta$GDP & Baseline & $\Delta$u & $\Delta$GDP \\
Variable &  &  &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.568 & 2.568 & 2.568 & 1.855 & 1.855 & 1.855 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.043 & 0.069 & 0.059 & 0.098 & 0.141 & 0.102 \\
$\sigma^{2}_e$ & 0.694 & 0.660 & 0.676 & -0.049 & -0.076 & -0.051 \\
$\sigma^{2}_b$ & 0.186 & 0.195 & 0.188 & 0.162 & 0.146 & 0.160 \\
% constant & 0.641 & 0.641 & 0.641 & 0.833 & 0.805 & 0.831 \\
% spell number & 0.017 & 0.027 & 0.023 & 0.053 & 0.076 & 0.055 \\
% duration dependence & 0.270 & 0.257 & 0.263 & -0.027 & -0.041 & -0.028 \\
% heterogeneity & 0.072 & 0.076 & 0.073 & 0.087 & 0.079 & 0.086 \\
N & 81,768 & 81,768 & 81,768 & 69,406 & 69,406 & 69,406 \\
Average duration (days) & 360.175 & 360.175 & 360.175 & 912.447 & 912.447 & 912.447 \\
\bottomrule
\end{tabular}



### 4.2 Time dummies

In [40]:
## i have commented this out as we dont use it in the paper
path_res = "./data/intermediate/germany/"

# Year Controls
pass_to_csv("038_decomp_yin",path_res)
ger_res_yin = pd.read_csv(path_res+"038_decomp_yin.csv",index_col=0).T
ger_res_yin = first_format(ger_res_yin,r'year')
# Quarter Controls
pass_to_csv("039_decomp_qin",path_res)
ger_res_qin = pd.read_csv(path_res+"039_decomp_qin.csv",index_col=0).T
ger_res_qin = first_format(ger_res_qin,r'quarter')

In [41]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"

# Year Controls
pass_to_csv("results2011_yearD_T_harm",path_res,path_out = path_out)
spa_res_yin = pd.read_csv(path_out+"results2011_yearD_T_harm.csv",index_col=0).T
spa_res_yin = first_format(spa_res_yin,r'year')
# Quarter Controls
pass_to_csv("results2011_qtD_T_harm",path_res,path_out = path_out)
spa_res_qin = pd.read_csv(path_out+"results2011_qtD_T_harm.csv",index_col=0).T
spa_res_qin = first_format(spa_res_qin,r'quarter')

In [42]:
format_res_table([spa_res_11,spa_res_yin,spa_res_qin])

,Baseline,year,quarter
Variable,,,
$\sigma^{2}_y$,1.86,1.86,1.86
$\sigma^{2}_c$,1.64,1.64,1.64
$\sigma^{2}_n$,0.10,0.58,0.59
$\sigma^{2}_e$,-0.05,-0.48,-0.49
$\sigma^{2}_b$,0.16,0.11,0.11
% constant,0.83,0.37,0.36
% spell number,0.05,0.31,0.32
% duration dependence,-0.03,-0.26,-0.26
% heterogeneity,0.09,0.06,0.06


In [43]:
print(format_res_table([ger_res,ger_res_yin,ger_res_qin,
                        spa_res_11,spa_res_yin,spa_res_qin]).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllllll}
\toprule
 & Baseline & year & quarter & Baseline & year & quarter \\
Variable &  &  &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.568 & 2.544 & 2.544 & 1.855 & 1.855 & 1.855 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.043 & 0.099 & 0.116 & 0.098 & 0.578 & 0.587 \\
$\sigma^{2}_e$ & 0.694 & 0.585 & 0.572 & -0.049 & -0.476 & -0.485 \\
$\sigma^{2}_b$ & 0.186 & 0.215 & 0.211 & 0.162 & 0.109 & 0.108 \\
% constant & 0.641 & 0.647 & 0.647 & 0.833 & 0.374 & 0.363 \\
% spell number & 0.017 & 0.039 & 0.045 & 0.053 & 0.311 & 0.317 \\
% duration dependence & 0.270 & 0.230 & 0.225 & -0.027 & -0.256 & -0.262 \\
% heterogeneity & 0.072 & 0.084 & 0.083 & 0.087 & 0.059 & 0.058 \\
N & 81,768 & 71,046 & 71,046 & 69,406 & 69,406 & 69,406 \\
Average duration (days) & 360.175 & 373.991 & 373.991 & 912.447 & 912.447 & 912.447 \\
\bottomrule
\end{tabular}



## 5. Pure Spells

In [44]:
path_res = "./data/intermediate/germany/"
path_res_jul = "./data/intermediate/germany/"

# Baseline decomp
pass_to_csv("031_decomp_2011",path_res)
ger_res = pd.read_csv(path_res+"031_decomp_2011.csv",index_col=0).T
ger_res = first_format(ger_res,'Germany')
# Only mini
pass_to_csv("040_decomp",path_res)
ger_res_mini = pd.read_csv(path_res+"040_decomp.csv",index_col=0).T
ger_res_mini = first_format(ger_res_mini,r'Only Mini-jobs')
# Only temp
pass_to_csv("041_decomp",path_res)
ger_res_temp = pd.read_csv(path_res+"041_decomp.csv",index_col=0).T
ger_res_temp = first_format(ger_res_temp,r'Only Temporary jobs')
# Mostly mini
pass_to_csv("042_decomp",path_res_jul)
ger_res_mmini = pd.read_csv(path_res_jul+"042_decomp.csv",index_col=0).T
ger_res_mmini = first_format(ger_res_mmini,r'Mostly Mini-jobs')
# SD in levels from 05_descriptives-5.log (tabstat days, >=25% mini sample)
ger_res_mmini['sig_y'] = 299889.6
# Mostly temp
pass_to_csv("043_decomp",path_res_jul)
ger_res_mtemp = pd.read_csv(path_res_jul+"043_decomp.csv",index_col=0).T
ger_res_mtemp = first_format(ger_res_mtemp,r'Mostly Temporary jobs')
# SD in levels from 05_descriptives-5.log (tabstat days, >=25% temp sample)
ger_res_mtemp['sig_y'] = 280340.4
# Mostly unemployment
pass_to_csv("044_decomp",path_res)
ger_res_munemp = pd.read_csv(path_res+"044_decomp.csv",index_col=0).T
ger_res_munemp = first_format(ger_res_munemp,r'Mostly Unemployment')
# SD in levels from 044_decomp.log (tabstat days, >=25% unemp sample)
ger_res_munemp['sig_y'] = 193515.6

In [45]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"

# Only T
pass_to_csv("results2011_onlyT_harm",path_res,path_out = path_out)
spa_res_t = pd.read_csv(path_out+"results2011_onlyT_harm.csv",index_col=0).T
spa_res_t = first_format(spa_res_t,r'Only Temporary Jobs')
# Only U
pass_to_csv("results2011_onlyU_harm",path_res,path_out = path_out)
spa_res_u = pd.read_csv(path_out+"results2011_onlyU_harm.csv",index_col=0).T
spa_res_u = first_format(spa_res_u,r'Only Unemployment')
# Some T (=<.45 of the spell)
pass_to_csv("results2011_someT_harm",path_res,path_out = path_out)
spa_res_mix = pd.read_csv(path_out+"results2011_someT_harm.csv",index_col=0).T
spa_res_mix = first_format(spa_res_mix,r'Mixed (<45% temp)')
# Mostly T (>.45 of the spell)
pass_to_csv("results2011_mostlyT_harm",path_res,path_out = path_out)
spa_res_mix2 = pd.read_csv(path_out+"results2011_mostlyT_harm.csv",index_col=0).T
spa_res_mix2 = first_format(spa_res_mix2,r'Mixed (>45% temp)')

In [46]:
print(format_res_table([ger_res,ger_res_temp,
                        spa_res_11,spa_res_t], include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & Germany & Only Temporary jobs & Baseline & Only Temporary Jobs \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.568 & 1.716 & 1.855 & 1.783 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.043 & 0.020 & 0.098 & -0.000 \\
$\sigma^{2}_e$ & 0.694 & -0.041 & -0.049 & -0.182 \\
$\sigma^{2}_b$ & 0.186 & 0.092 & 0.162 & 0.320 \\
% constant & 0.641 & 0.911 & 0.833 & 0.718 \\
% spell number & 0.017 & 0.011 & 0.053 & -0.000 \\
% duration dependence & 0.270 & -0.024 & -0.027 & -0.102 \\
% heterogeneity & 0.072 & 0.054 & 0.087 & 0.180 \\
N & 81,768 & 3,633 & 69,406 & 6,851 \\
Average duration (days) & 360.175 & 524.052 & 912.447 & 391.880 \\
Std. duration (days) & 494.538 & 509.124 & 1047.165 & 473.679 \\
\bottomrule
\end{tabular}



In [47]:
print(format_res_table([ger_res_mtemp,spa_res_mix2], include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lll}
\toprule
 & Mostly Temporary jobs & Mixed (>45% temp) \\
Variable &  &  \\
\midrule
$\sigma^{2}_y$ & 1.305 & 1.505 \\
$\sigma^{2}_c$ & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.007 & 0.034 \\
$\sigma^{2}_e$ & -0.398 & -0.410 \\
$\sigma^{2}_b$ & 0.051 & 0.237 \\
% constant & 0.650 & 0.548 \\
% spell number & 0.005 & 0.022 \\
% duration dependence & -0.305 & -0.273 \\
% heterogeneity & 0.039 & 0.157 \\
N & 5,342 & 27,191 \\
Average duration (days) & 588.105 & 850.299 \\
Std. duration (days) & 529.472 & 960.101 \\
\bottomrule
\end{tabular}



In [48]:
format_res_table([ger_res,ger_res_mini,ger_res_temp,
                        spa_res_11,spa_res_t,spa_res_u], include_sd=True)

,Germany,Only Mini-jobs,Only Temporary jobs,Baseline,Only Temporary Jobs,Only Unemployment
Variable,,,,,,
$\sigma^{2}_y$,2.57,2.28,1.72,1.86,1.78,1.86
$\sigma^{2}_c$,1.64,1.64,1.64,1.64,1.64,1.64
$\sigma^{2}_n$,0.04,0.04,0.02,0.10,-0.00,0.03
$\sigma^{2}_e$,0.69,0.50,-0.04,-0.05,-0.18,-0.27
$\sigma^{2}_b$,0.19,0.08,0.09,0.16,0.32,0.46
% constant,0.64,0.72,0.91,0.83,0.72,0.60
% spell number,0.02,0.02,0.01,0.05,-0.00,0.01
% duration dependence,0.27,0.22,-0.02,-0.03,-0.10,-0.14
% heterogeneity,0.07,0.04,0.05,0.09,0.18,0.25


In [49]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"
years = [2011, 2005, 1999, 1992]
tables_mixed,tables_mixed2 = [], []

for year in years:
    pass_to_csv("results{}_someT_harm".format(year),path_res,path_out = path_out)
    spa_res_mix = pd.read_csv(path_out+"results{}_someT_harm.csv".format(year),index_col=0).T
    spa_res_mix = first_format(spa_res_mix,year)
    tables_mixed.append(spa_res_mix)

    pass_to_csv("results{}_mostlyT_harm".format(year),path_res,path_out = path_out)
    spa_res_mix = pd.read_csv(path_out+"results{}_mostlyT_harm.csv".format(year),index_col=0).T
    spa_res_mix = first_format(spa_res_mix,year)
    tables_mixed2.append(spa_res_mix)

In [50]:
print(format_res_table(tables_mixed, include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & 2011 & 2005 & 1999 & 1992 \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 2.247 & 2.357 & 2.435 & 2.391 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.131 & 0.017 & 0.015 & 0.001 \\
$\sigma^{2}_e$ & 0.130 & 0.289 & 0.311 & 0.162 \\
$\sigma^{2}_b$ & 0.341 & 0.406 & 0.465 & 0.582 \\
% constant & 0.732 & 0.698 & 0.676 & 0.688 \\
% spell number & 0.058 & 0.007 & 0.006 & 0.001 \\
% duration dependence & 0.058 & 0.123 & 0.128 & 0.068 \\
% heterogeneity & 0.152 & 0.172 & 0.191 & 0.244 \\
N & 20,648 & 38,598 & 51,471 & 96,779 \\
Average duration (days) & 823.910 & 963.476 & 1053.930 & 803.132 \\
Std. duration (days) & 1016.173 & 1190.300 & 1264.977 & 1178.020 \\
\bottomrule
\end{tabular}



In [51]:
print(pd.concat((format_res_table(tables_mixed2, include_sd=True),format_res_table(tables_mixed, include_sd=True)),axis=1).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllllllll}
\toprule
 & 2011 & 2005 & 1999 & 1992 & 2011 & 2005 & 1999 & 1992 \\
Variable &  &  &  &  &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 1.505 & 1.473 & 1.437 & 1.512 & 2.247 & 2.357 & 2.435 & 2.391 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.034 & 0.000 & -0.000 & 0.003 & 0.131 & 0.017 & 0.015 & 0.001 \\
$\sigma^{2}_e$ & -0.410 & -0.395 & -0.409 & -0.316 & 0.130 & 0.289 & 0.311 & 0.162 \\
$\sigma^{2}_b$ & 0.237 & 0.223 & 0.200 & 0.180 & 0.341 & 0.406 & 0.465 & 0.582 \\
% constant & 0.548 & 0.580 & 0.576 & 0.670 & 0.732 & 0.698 & 0.676 & 0.688 \\
% spell number & 0.022 & 0.000 & -0.000 & 0.002 & 0.058 & 0.007 & 0.006 & 0.001 \\
% duration dependence & -0.273 & -0.268 & -0.284 & -0.209 & 0.058 & 0.123 & 0.128 & 0.068 \\
% heterogeneity & 0.157 & 0.151 & 0.139 & 0.119 & 0.152 & 0.172 & 0.191 & 0.244 \\
N & 27,191 & 57,379 & 80,256 & 88,114 & 20,648 & 38,598 & 51,471 & 96,779 \\
Average duration (days) & 850.

## 6. Only the Slow (9 year gap)

In [52]:
path_res = "./data/intermediate/spain/results/"
path_out = "./data/intermediate/spain/results/"

pass_to_csv("Over9years1992_T_harm",path_res,path_out = path_out)
spa_res_slow = pd.read_csv(path_out+"Over9years1992_T_harm.csv",index_col=0).T
spa_res_slow = first_format(spa_res_slow,"9-year gap")



pass_to_csv("results2011_T_harm_quits",path_res,path_out = path_out)
spa_res_quits = pd.read_csv(path_out+"results2011_T_harm_quits.csv",index_col=0).T
spa_res_quits = first_format(spa_res_quits,"Quits only")

In [53]:
print(format_res_table((tables[-1],spa_res_slow), include_sd=True))

                            1992 9-year gap
Variable                                   
$\sigma^{2}_y$              2.08       2.14
$\sigma^{2}_c$              1.64       1.64
$\sigma^{2}_n$              0.00       0.34
$\sigma^{2}_e$              0.10       0.11
$\sigma^{2}_b$              0.33       0.05
% constant                  0.79       0.77
% spell number              0.00       0.16
% duration dependence       0.05       0.05
% heterogeneity             0.16       0.03
N                        215,907     41,576
Average duration (days)   927.49   1,750.25
Std. duration (days)    1,187.57   1,739.01


In [54]:
print(format_res_table((tables[0],spa_res_quits,tables[-1],spa_res_slow), include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & 2011 & Quits only & 1992 & 9-year gap \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 1.855 & 2.015 & 2.081 & 2.142 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.098 & 0.086 & 0.001 & 0.336 \\
$\sigma^{2}_e$ & -0.049 & 0.062 & 0.103 & 0.108 \\
$\sigma^{2}_b$ & 0.162 & 0.222 & 0.332 & 0.054 \\
% constant & 0.833 & 0.816 & 0.791 & 0.768 \\
% spell number & 0.053 & 0.043 & 0.000 & 0.157 \\
% duration dependence & -0.027 & 0.031 & 0.050 & 0.050 \\
% heterogeneity & 0.087 & 0.110 & 0.159 & 0.025 \\
N & 69,406 & 6,235 & 215,907 & 41,576 \\
Average duration (days) & 912.447 & 831.426 & 927.491 & 1750.248 \\
Std. duration (days) & 1047.165 & 1007.353 & 1187.574 & 1739.006 \\
\bottomrule
\end{tabular}



In [55]:
print(format_res_table((tables[0],spa_res_quits,tables[-1],spa_res_slow), include_sd=True).to_latex(escape=False,float_format="%.3f"))

\begin{tabular}{lllll}
\toprule
 & 2011 & Quits only & 1992 & 9-year gap \\
Variable &  &  &  &  \\
\midrule
$\sigma^{2}_y$ & 1.855 & 2.015 & 2.081 & 2.142 \\
$\sigma^{2}_c$ & 1.645 & 1.645 & 1.645 & 1.645 \\
$\sigma^{2}_n$ & 0.098 & 0.086 & 0.001 & 0.336 \\
$\sigma^{2}_e$ & -0.049 & 0.062 & 0.103 & 0.108 \\
$\sigma^{2}_b$ & 0.162 & 0.222 & 0.332 & 0.054 \\
% constant & 0.833 & 0.816 & 0.791 & 0.768 \\
% spell number & 0.053 & 0.043 & 0.000 & 0.157 \\
% duration dependence & -0.027 & 0.031 & 0.050 & 0.050 \\
% heterogeneity & 0.087 & 0.110 & 0.159 & 0.025 \\
N & 69,406 & 6,235 & 215,907 & 41,576 \\
Average duration (days) & 912.447 & 831.426 & 927.491 & 1750.248 \\
Std. duration (days) & 1047.165 & 1007.353 & 1187.574 & 1739.006 \\
\bottomrule
\end{tabular}



---
## Export: write all paper tables to `output/tables/results_tables.tex`

In [56]:
## ── Export all paper tables to output/tables/results_tables.tex ───────────────────────
#
# NOTE: several cells reuse the names `ger_res` and `spa_res_11` with
# different column labels for different tables.  At the end of the notebook:
#   ger_res      -> Germany baseline data, label = 'Germany'  (set in cell 57)
#   spa_res_11   -> Spain   baseline data, label = 'Baseline' (set in cell 48)
# `relabel()` is used below to attach the right column header without touching
# any earlier cell.

import re

def relabel(s, name):
    """Return a copy of Series s with .name changed to `name`."""
    s = s.copy()
    s.name = name
    return s

def fix_latex(s):
    """
    Post-process a to_latex() string to match paper formatting:
      1. Escape % in row labels so they are not treated as LaTeX comments.
      2. Insert \\midrule between the sigma^2_b row and the share rows.
      3. Insert \\hline \\hline before the N row.
    """
    # 1. Escape % in known percentage-share row labels
    for row in ['constant', 'spell number', 'duration dependence',
                'heterogeneity', 'controls']:
        s = s.replace(f'% {row} &', f'\\% {row} &')

    # 2. Insert \midrule after the $\sigma^{2}_b$ row
    lines = s.split('\n')
    out = []
    for line in lines:
        out.append(line)
        if '$\\sigma^{2}_b$' in line and '&' in line:
            out.append('\\midrule')
    s = '\n'.join(out)

    # 3. Insert \hline \hline before the N row
    lines = s.split('\n')
    out = []
    for line in lines:
        if re.match(r'^N\s*&', line.strip()):
            out.append('\\hline \\hline')
        out.append(line)
    s = '\n'.join(out)

    return s

def apply_header(s, country_groups=None, col_labels=None):
    """
    Replace the auto-generated column header line with paper-format headers.

    country_groups : list of (country_name, n_cols), e.g. [('Germany',2),('Spain',2)]
                     When given, a \\multicolumn country row is inserted first.
    col_labels     : list of per-column labels for the sub-header row,
                     e.g. ['female','male','female','male'].
                     When given, this becomes the 'Variable & ... \\\\' row.

    Tables that already carry 'Germany'/'Spain' directly in the pandas header
    (tab:results_main, no-outliers) and single-country tables should NOT call
    this function.
    """
    lines = s.split('\n')
    out = []
    skip_next = False
    for line in lines:
        if skip_next:
            skip_next = False   # discard the old auto-generated column header
            continue
        out.append(line)
        if line.strip() == '\\toprule':
            skip_next = True
            # Row 1 (optional): \multicolumn country grouping
            if country_groups is not None:
                n_countries = len(country_groups)
                mc_parts = []
                for j, (country, n) in enumerate(country_groups):
                    fmt = 'c|' if j < n_countries - 1 else 'c'
                    mc_parts.append(f'\\multicolumn{{{n}}}{{{fmt}}}{{{country}}}')
                out.append(' & ' + ' & '.join(mc_parts) + ' \\\\[5pt]')
            # Row 2: per-column labels
            if col_labels is not None:
                out.append('Variable & ' + ' & '.join(col_labels) + ' \\\\')
    # Clean old latex headers
    out = [line for line in out
         if not (line.strip().startswith('Variable') and all(
             p.strip() == '' for p in line.strip().rstrip('\\').split('&')[1:]))]
    return '\n'.join(out)

def tex(tables, include_sd=False):
    """Format, convert to LaTeX, and apply paper-style fixes."""
    return fix_latex(
        format_res_table(tables, include_sd=include_sd).to_latex(escape=False, float_format="%.3f")
    )

sections = []

# ── 1. Baseline (tab:results_main) ──────────────────────────────────────────
# Columns are already labelled 'Germany' / 'Spain' — no header change needed.
# SD in levels available for both countries.
sections.append("% tab:results_main -- Baseline decomposition\n" +
    tex([relabel(ger_res, 'Germany'), relabel(spa_res_11, 'Spain')], include_sd=True))

# ── 2. Temporary contracts / mostly-temp (tab:results_mixed) ─────────────────
# Generated labels: 'Mostly Temporary jobs' / 'Mixed (>45% temp)'
# Paper labels:     'Mostly Temp'           / 'Mostly Temp'
# SD in levels available for both countries.
t = tex([ger_res_mtemp, spa_res_mix2], include_sd=True)
t = apply_header(t,
    country_groups=[('Germany', 1), ('Spain', 1)],
    col_labels=['Mostly Temp', 'Mostly Temp'])
sections.append("% tab:results_mixed -- Mostly-temporary contracts\n" + t)

# ── 3. By sex (decomp_sex) ────────────────────────────────────────────────────
t = tex([ger_res_fem, ger_res_mal, spa_res_fem, spa_res_mal], include_sd=True)
t = apply_header(t,
    country_groups=[('Germany', 2), ('Spain', 2)],
    col_labels=['female', 'male', 'female', 'male'])
sections.append("% decomp_sex -- Decomposition by sex\n" + t)

# ── 4. Business cycle controls (tab:results_spain_BC) ────────────────────────
# ger_res is currently labeled 'Germany'; relabel to 'Baseline' for this table.
# spa_res_11 is already labeled 'Baseline' (set in cell 48).
t = tex([relabel(ger_res, 'Baseline'), ger_res_unemp, ger_res_gdp,
         spa_res_11,                   spa_res_unemp, spa_res_gdp])
# Rename '% spell number' -> '% spell number and BC controls' for this table only
t = t.replace(r'\% spell number &', r'\% spell number and BC controls &')
t = apply_header(t,
    country_groups=[('Germany', 3), ('Spain', 3)],
    col_labels=['Baseline', r'$\Delta$u', r'$\Delta$GDP',
                'Baseline', r'$\Delta$u', r'$\Delta$GDP'])
sections.append("% tab:results_spain_BC -- Business cycle controls\n" + t)

# ── 5. No outliers (appendix, no label in paper) ─────────────────────────────
# Columns are already 'Germany' / 'Spain' — no header change needed.
sections.append("% Appendix: No outliers\n" +
    tex([ger_res_noout, spa_res_11_noout], include_sd=True))

# ── 6. Age groups (appendix, no label) ───────────────────────────────────────
t = tex([ger_res_25_35, ger_res_35_55, spa_res_25_35, spa_res_35_55], include_sd=True)
t = apply_header(t,
    country_groups=[('Germany', 2), ('Spain', 2)],
    col_labels=['25-35', '35-55', '25-35', '35-55'])
sections.append("% Appendix: Decomposition by age group (25-35 / 35-55)\n" + t)

# ── 7. Age controls (appendix, no label) ─────────────────────────────────────
# ger_res_age and spa_res_age were set in cells 43-44 and not overwritten.
# ger_res needs 'Baseline'; spa_res_11 is already 'Baseline'.
t = tex([relabel(ger_res, 'Baseline'), ger_res_age,
         spa_res_11,                   spa_res_age])
t = apply_header(t,
    country_groups=[('Germany', 2), ('Spain', 2)],
    col_labels=['Baseline', 'Age controls', 'Baseline', 'Age controls'])
sections.append("% Appendix: Decomposition with age controls\n" + t)

# ── 8. Education (appendix, no label) ────────────────────────────────────────
# Single-country tables — no country header needed; column labels already match.
sections.append("% Appendix: Germany -- by educational level\n" +
    tex(res_ger_edu, include_sd=True))

sections.append("% Appendix: Spain -- by educational level\n" +
    tex(res_spa_edu, include_sd=True))

# ── 9. Pure temporary spells (tab:purespells, appendix) ──────────────────────
# ger_res needs 'Baseline'; spa_res_11 is already 'Baseline'.
t = tex([relabel(ger_res, 'Baseline'), ger_res_temp,
         relabel(spa_res_11, 'Baseline'), spa_res_t], include_sd=True)
t = apply_header(t,
    country_groups=[('Germany', 2), ('Spain', 2)],
    col_labels=['Baseline', 'Only Temporary jobs',
                'Baseline', 'Only Temporary Jobs'])
sections.append("% tab:purespells -- Pure / only-temporary spells\n" + t)

# ── 10. Spain time windows (tab:results_spain_sample, appendix) ──────────────
# Spain-only, no country header. Generated labels '2011'/'2005'/'1999'/'1992'
# → paper uses 'Baseline'/'2005-'/'1999-'/'1992-'.
t = tex(tables, include_sd=True)
t = apply_header(t, col_labels=['Baseline', '2005-', '1999-', '1992-'])
sections.append("% tab:results_spain_sample -- Spain, different time windows\n" + t)


# ── 11. Spain mixed spells (tab:spain_mixed, appendix) ────────────────────
# Baseline | Mostly Temporary | Mostly Unemployment (3 columns)
# NOTE: spa_res_mix and spa_res_mix2 are stale by this point (overwritten in the time-window loop).
# We reload the 2011 files directly to ensure correct vintage.
_path_res_spa = "./data/intermediate/spain/results/"
_path_out_spa = "./data/intermediate/spain/results/"

pass_to_csv("results2011_mostlyT_harm", _path_res_spa, path_out=_path_out_spa)
_spa_mostly_t = pd.read_csv(_path_out_spa + "results2011_mostlyT_harm.csv", index_col=0).T
_spa_mostly_t = first_format(_spa_mostly_t, r'Mostly Temporary')

pass_to_csv("results2011_someT_harm", _path_res_spa, path_out=_path_out_spa)
_spa_some_t = pd.read_csv(_path_out_spa + "results2011_someT_harm.csv", index_col=0).T
_spa_some_t = first_format(_spa_some_t, r'Mostly Unemployment')

t = tex([relabel(spa_res_11, 'Baseline'), _spa_mostly_t, _spa_some_t], include_sd=True)
t = apply_header(t,
    country_groups=[('Spain', 3)],
    col_labels=['Baseline', 'Mostly Temporary', 'Mostly Unemployment'])
sections.append("% tab:spain_mixed -- Spain mixed spells\n" + t)

# ── 12. Germany time windows (tab:results_germany_sample, appendix) ──────────────
# Germany-only, no country header. Generated labels '2016/2014/2011'
tables_ger_windows = tables_1416

t = tex(tables_ger_windows, include_sd=True)
t = apply_header(t, col_labels=['Baseline', '2014-', '2016-'])
sections.append("% tab:results_germany_sample -- Germany, different time windows\n" + t)

# ── 13. Germany mixed spells (tab:germany_mixed, appendix) ──────────────────
# Baseline | Mostly Mini | Mostly Temporary | Mostly Unemployment (4 columns)
t = tex([relabel(ger_res, 'Baseline'), ger_res_mmini, ger_res_mtemp, ger_res_munemp], include_sd=True)
t = apply_header(t,
    country_groups=[('Germany', 4)],
    col_labels=['Baseline', 'Mostly Mini', 'Mostly Temporary', 'Mostly Unemployment'])
sections.append("% tab:germany_mixed -- Germany mixed spells\n" + t)

# ── Write file ────────────────────────────────────────────────────────────────
output_tex = "./output/tables/results_tables.tex"
with open(output_tex, 'w', encoding='utf-8') as f:
    f.write("% Results Tables\n")
    f.write("% Auto-generated by 'Results_Tables.ipynb'\n")
    f.write("% Do not edit manually - re-run the notebook to update.\n\n")
    f.write("\n\n".join(sections))
    f.write("\n")

print(f"Saved {len(sections)} tables to {output_tex}")

Saved 14 tables to ./output/tables/results_tables.tex
